In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        
        assert self.head_dim * heads == embed_size, "Embed size需要被heads整除"
        
        self.values = nn.Linear(self.head_dim, self.head_dim)
        self.keys = nn.Linear(self.head_dim, self.head_dim)
        self.queries = nn.Linear(self.head_dim, self.head_dim)
        self.fc_out = nn.Linear(embed_size, embed_size)
        
    def forward(self, values, keys, query):
        batch_size = query.shape[0]
        
        # 分割成多头
        values = values.reshape(batch_size, -1, self.heads, self.head_dim)
        keys = keys.reshape(batch_size, -1, self.heads, self.head_dim)
        queries = query.reshape(batch_size, -1, self.heads, self.head_dim)
        
        # 计算注意力
        energy = torch.einsum("bqhd,bkhd->bhqk", [queries, keys])
        attention = F.softmax(energy / (self.embed_size ** (1/2)), dim=3)
        
        out = torch.einsum("bhql,blhd->bqhd", [attention, values])
        out = out.reshape(batch_size, -1, self.embed_size)
        
        out = self.fc_out(out)
        return out

In [15]:
import torch

In [22]:
a = torch.randn(2,2,4)
a

tensor([[[ 0.4704,  0.1466, -1.1279,  0.6628],
         [ 0.4987, -0.6083, -0.2234, -2.0504]],

        [[-0.4013,  0.6660, -1.8718,  2.0948],
         [ 0.9349,  0.7660,  0.4326,  0.1366]]])

In [23]:
a = a.reshape(2,2,2,2)
a

tensor([[[[ 0.4704,  0.1466],
          [-1.1279,  0.6628]],

         [[ 0.4987, -0.6083],
          [-0.2234, -2.0504]]],


        [[[-0.4013,  0.6660],
          [-1.8718,  2.0948]],

         [[ 0.9349,  0.7660],
          [ 0.4326,  0.1366]]]])